In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
 
pd.set_option("future.no_silent_downcasting", True)
 
from laparoscopy_helpers.data_cleaning import to_snake_case, clean_surgical_df
from poor_patient_helpers.data_cleaning import load_all_endo, load_all_mercy, load_all_safe, build_final

In [2]:
TB = pd.read_excel(
    "../Nkhoma_data/theatre_book_data/combined_clean.xlsx",
    engine="openpyxl"
)

In [3]:
base_path = "../Nkhoma_data/poor_patients_funds_data"

endo  = load_all_endo(f"{base_path}/Endoscopy Ledger")
mercy = load_all_mercy(f"{base_path}/Mercy Fund")
safe  = load_all_safe(f"{base_path}/SAFE")

final = build_final(TB, mercy, safe, verbose=True)

✓ endo  22 May               → 14 rows
✓ endo  22 June              → 7 rows
✓ endo  22 Aug               → 14 rows
✓ endo  22 Sept              → 6 rows
✓ endo  22 Oct               → 5 rows
✓ endo  22 Nov               → 7 rows
✓ endo  22 Dec               → 6 rows
✓ endo  23 JAN               → 9 rows
✓ endo  23 FEB               → 11 rows
✓ endo  23 MAR               → 9 rows
✓ endo  23 APR               → 16 rows
✓ endo  23 MAY               → 6 rows
✓ endo  23 SEPT              → 21 rows
✓ endo  23 OCT               → 4 rows
✓ endo  23 NOV               → 14 rows
✓ endo  23 DEC               → 9 rows
✓ endo  2024 FULL LIST       → 124 rows
✓ endo  January              → 15 rows
✓ endo  February             → 15 rows
✓ endo  March                → 20 rows
✓ endo  April                → 20 rows
✓ endo  May                  → 20 rows
✓ endo  June                 → 25 rows
✓ endo  July                 → 20 rows
✓ endo  August               → 28 rows
✓ endo  September            → 27 

In [4]:
final.columns

Index(['fund', 'match_status', 'match_type', 'fuzzy_score', 'unmatched_reason',
       'cross_fund_flag', 'fund_name', 'patient_number', 'age', 'gender',
       'fund_admission_date', 'fund_date', 'theatre_date', 'date_diff_days',
       'total_bill_mk', 'copay_mk', 'fund_covered_mk', 'mercy_fund_mk',
       'discount_mk', 'safe_ask_usd', 'safe_fund_mk', 'invoice_number',
       'fund_diagnosis', 'fund_surgery', 'fund_source', 'admission_type',
       'theatre_index', 'theatre_name', 'department', 'tb_diagnosis',
       'tb_procedure', 'procedure_free_text', 'tb_surgery_type',
       'indication_for_surgery', 'urgency', 'surgery_severity', 'asascore',
       'days_since_last_mercy', 'possible_duplicate_mercy',
       'days_since_last_safe', 'possible_duplicate_safe'],
      dtype='object')

In [6]:
matched = final[final["match_status"] == "matched"].copy()
unmatched = final[final["match_status"] == "unmatched"].copy()

# ── 1. Confirm Ortho + Gen_Surg only for SAFE and Mercy ──────────────
print("=== Q1: Department distribution (matched only) ===")
print(matched.groupby(["fund", "department"]).size().unstack(fill_value=0))

# ── 2. Cross-fund double-billing check ───────────────────────────────
# Same patient_number appearing in both mercy and safe
mercy_ids = set(final[final["fund"]=="mercy"]["patient_number"].dropna().astype(str))
safe_ids  = set(final[final["fund"]=="safe"]["patient_number"].dropna().astype(str))
endo_ids  = set(endo["patient_number"].dropna().astype(str))

print(f"\n=== Q2: Cross-fund overlap (should be ~0) ===")
print(f"Mercy ∩ Safe:  {len(mercy_ids & safe_ids)}")
print(f"Mercy ∩ Endo:  {len(mercy_ids & endo_ids)}")
print(f"Safe  ∩ Endo:  {len(safe_ids  & endo_ids)}")

if mercy_ids & safe_ids:
    overlap = sorted(mercy_ids & safe_ids)
    print(f"Overlapping IDs: {overlap[:10]}")

# ── 3. One admission → many procedures ───────────────────────────────
print("\n=== Q3: Procedures per fund entry (1 fund row → N theatre rows) ===")
multi = (matched.groupby(["fund", "fund_name", "fund_admission_date"])
         ["theatre_index"].count()
         .reset_index(name="n_procedures"))
print(multi["n_procedures"].value_counts().sort_index())
print("\nSample multi-procedure cases:")
print(multi[multi["n_procedures"] > 1]
      .sort_values("n_procedures", ascending=False)
      .head(10).to_string())

# ── 4. Coverage summary: how many TB procedures were covered ──────────
print("\n=== Q4: Theatre book coverage ===")
tb_total = len(TB[TB["department"].isin(["Gen_Surg", "Ortho"])])
tb_covered = matched["theatre_index"].nunique()
print(f"TB Gen_Surg + Ortho procedures total:   {tb_total}")
print(f"Covered by a fund (unique TB entries):  {tb_covered}")
print(f"Coverage rate:                          {tb_covered/tb_total:.1%}")

print("\nCovered by fund:")
print(matched.groupby("fund")["theatre_index"].nunique())

# ── 5. Procedure breakdown for covered cases ─────────────────────────
print("\n=== Q5: What procedures were covered? (tb_diagnosis) ===")
proc_table = (matched.groupby(["fund", "tb_diagnosis"])
              .size()
              .reset_index(name="n")
              .pivot_table(index="tb_diagnosis", columns="fund", 
                           values="n", fill_value=0)
              .sort_values("mercy", ascending=False))
print(proc_table.head(20).to_string())

# ── 6. Unmatched breakdown ────────────────────────────────────────────
print("\n=== Q6: Unmatched fund entries ===")
print(unmatched.groupby(["fund", "unmatched_reason"]).size().unstack(fill_value=0))

unmatched_in_range = unmatched[unmatched["unmatched_reason"] == "not_in_TB"]
print(f"\nUnmatched within TB date range: {len(unmatched_in_range)}")
print("These are likely endoscopy-only or missing TB entries")

# ── 7. Financial summary ──────────────────────────────────────────────
print("\n=== Q7: Financial summary (matched cases only) ===")
fin = (matched.drop_duplicates(subset=["fund", "fund_name", "fund_admission_date"])
       .groupby("fund")
       .agg(
           n_cases       =("fund_name",       "count"),
           total_billed  =("total_bill_mk",   "sum"),
           total_copay   =("copay_mk",        "sum"),
           total_covered =("fund_covered_mk", "sum"),
       ))
fin["avg_covered_per_case"] = (fin["total_covered"] / fin["n_cases"]).round(0)
print(fin.to_string())

=== Q1: Department distribution (matched only) ===
department  Gen_Surg  Ortho
fund                       
mercy            703     43
safe             263    128

=== Q2: Cross-fund overlap (should be ~0) ===
Mercy ∩ Safe:  15
Mercy ∩ Endo:  7
Safe  ∩ Endo:  3
Overlapping IDs: ['133337', '15031', '19669', '20643', '372415', '373557', '383000', '383228', '384127', '389772']

=== Q3: Procedures per fund entry (1 fund row → N theatre rows) ===
n_procedures
1    851
2     77
3     13
4      4
Name: count, dtype: int64

Sample multi-procedure cases:
      fund          fund_name fund_admission_date  n_procedures
505  mercy     PEMPHERO MARKO          2024-03-15             4
496  mercy     OLASIO ANTONIO          2024-05-13             4
629  mercy    YESAYA LATUMELO          2023-08-14             4
495  mercy    OLASIO ANTHONYO          2024-04-05             4
9    mercy  ALBERT KANKHUMBWA          2023-08-23             3
256  mercy     GENESIS MALAKI          2024-01-27             3


In [8]:
# ── Investigate cross-fund overlaps ───────────────────────────────────
overlap_ids = mercy_ids & safe_ids
overlap_cases = final[final["patient_number"].astype(str).isin(overlap_ids)][
    ["fund", "patient_number", "fund_name", "fund_admission_date",
     "total_bill_mk", "fund_covered_mk", "department"]
].sort_values(["patient_number", "fund_admission_date"])
print("=== Cross-fund patients (mercy AND safe) ===")
print(overlap_cases.to_string())

# ── Flag OLASIO duplicate ─────────────────────────────────────────────
print("\n=== OLASIO entries — likely same person? ===")
print(final[final["fund_name"].str.contains("OLASIO", na=False)][
    ["fund", "fund_name", "patient_number", "fund_admission_date",
     "theatre_index", "theatre_name", "tb_diagnosis"]
].to_string())

# ── SAFE financial note ───────────────────────────────────────────────
print("\n=== SAFE: safe_fund_mk distribution ===")
safe_fin = final[final["fund"] == "safe"].drop_duplicates(
    subset=["fund_name", "fund_admission_date"])
print(f"Positive (SAFE covered more than billed): "
      f"{(safe_fin['safe_fund_mk'] > 0).sum()}")
print(f"Negative (SAFE fixed price < actual bill): "
      f"{(safe_fin['safe_fund_mk'] < 0).sum()}")
print(f"Null: {safe_fin['safe_fund_mk'].isna().sum()}")
print(f"\nMedian safe_ask_usd: ${safe_fin['safe_ask_usd'].median():.0f}")
print(f"Median total_bill_mk: {safe_fin['total_bill_mk'].median():,.0f} MK")

=== Cross-fund patients (mercy AND safe) ===
       fund patient_number            fund_name fund_admission_date  total_bill_mk  fund_covered_mk department
990   mercy           5882          SONGO LINJE          2025-02-25       370000.0         320000.0       None
1994   safe           5882          SONGO LINJE          2025-05-29       882504.0         -38826.0       None
963   mercy           6086       ATANAZIO BANDA          2025-02-23       203400.0          63400.0       None
1970   safe           6086       ATANAZIO BANDA          2025-03-06       882504.0         142844.0       None
1958   safe           6960        ARMANDO KOREA          2025-03-13       882504.0         101944.0       None
1144  mercy           6960       ALUMANDO KOREA          2025-05-06       175700.0         125700.0       None
1960   safe           7877  CHIMTHUNZI KUTCHIKA          2025-03-20       882504.0          73744.0       None
1073  mercy           7877   CHINTHUZI KUTCHIKA          2025-03-28